In [ ]:
import pandas as pd
import pycountry

In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
            "prepare_political_stability",
    )

### File paths

In [ ]:
political_stability_fn = snakemake.input.political_stability
bus_locations_fn = snakemake.input.bus_locations

# Output
political_stability_clustered_fn = snakemake.output.political_stability

# config
config = snakemake.config

### Get Political Stability data

In [ ]:
# Read the CSV data, skipping the first 4 rows of metadata
# Column headers are: Country Name, Country Code, Indicator Name, Indicator Code, then years
ps = pd.read_csv(political_stability_fn, skiprows=4)

In [ ]:
# Display first rows to understand structure
ps.head()

### Extract 2023 values

In [ ]:
# Keep only Country Name and Country Code (alpha_3), and the 2023 column
# Some rows might not have 2023 data, so we need to check
year_columns = [str(year) for year in range(1960, 2026)]

# Find the latest available value for each country (backwards from 2023)
ps_values = []

for idx, row in ps.iterrows():
    country_name = row['Country Name']
    country_code = row['Country Code']
    
    # Find the latest available value from 2023 backwards
    latest_value = None
    latest_year = None
    
    for year in range(2023, 1959, -1):
        year_str = str(year)
        if year_str in row.index:
            val = row[year_str]
            # Check if value exists and is not NaN
            if pd.notna(val):
                latest_value = val
                latest_year = year
                break
    
    if latest_value is not None:
        ps_values.append({
            'country_name': country_name,
            'country_code': country_code,
            'political_stability': latest_value,
            'data_year': latest_year
        })

ps_latest = pd.DataFrame(ps_values)
ps_latest.head(10)

### Cluster per region

Map each country in the data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config file
regions = config['regions']

In [ ]:
country_name_corrections = {
    "Democratic Republic of the Congo": "Congo, The Democratic Republic of the",
    "Republic of the Congo": "Republic of the Congo",
    "Kosovo": "Republic of Kosovo",  # pycountry not supported
    "Russia": "Russian Federation",
    "Turkey": "Türkiye",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Tanzania": "United Republic of Tanzania",
    "Bolivia": "Plurinational State of Bolivia",
    "Vietnam": "Viet Nam",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Taiwan": "Taiwan, Province of China",
    "Laos": "Lao People's Democratic Republic",
    "Brunei": "Brunei Darussalam",
    "Equatorial French Guiana": "French Guiana",
    "Syria": "Syrian Arab Republic",   
    "Palestine": "Palestine, State of",
    "Moldova": "Republic of Moldova",
}

In [ ]:
# Apply country name corrections to the regions dict
for region, countries in regions.items():
    for i, country in enumerate(countries):
        if country in country_name_corrections:
            countries[i] = country_name_corrections[country]

In [ ]:
# Build a mapping from country name to ISO_A3 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_3
    # Add common names
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_3

# Build a mapping from ISO_A3 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        # Some country names may have extra text (e.g., "Togo + Algeria"), handle them simply
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

In [ ]:
# Map each row in political stability data to its region
ps_latest['region'] = ps_latest['country_code'].map(lambda iso: iso_to_region.get(iso, 'Other'))

In [ ]:
# Filter for relevant columns
ps_latest = ps_latest[["political_stability", "region"]]
ps_latest.head()

In [ ]:
# Aggregate by region: calculate mean political stability for each region
ps_by_region = ps_latest.groupby('region').mean(numeric_only=True)
ps_by_region

### Save clustered Political Stability data

In [ ]:
ps_by_region.to_csv(political_stability_clustered_fn)